In [1]:
import os
import urllib.request

# Đường dẫn này nằm trong Docker nhưng nối ra máy thật của em
JAR_DIR = "/home/iceberg/notebooks/jars_permanent"
if not os.path.exists(JAR_DIR):
    os.makedirs(JAR_DIR)

libs = {
    "hadoop-aws-3.3.4.jar": "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
    "aws-java-sdk-bundle-1.12.262.jar": "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar"
}

print(f"🚀 Đang tải thư viện về ổ cứng máy em: {JAR_DIR}")

for name, url in libs.items():
    file_path = os.path.join(JAR_DIR, name)
    if not os.path.exists(file_path) or os.path.getsize(file_path) < 1024:
        print(f"⬇️ Đang tải {name}...")
        try:
            urllib.request.urlretrieve(url, file_path)
            print(f"✅ Tải xong: {name}")
        except Exception as e:
            print(f"❌ Lỗi tải {name}: {e}")
    else:
        print(f"✅ Đã có sẵn: {name}")

🚀 Đang tải thư viện về ổ cứng máy em: /home/iceberg/notebooks/jars_permanent
✅ Đã có sẵn: hadoop-aws-3.3.4.jar
✅ Đã có sẵn: aws-java-sdk-bundle-1.12.262.jar


In [2]:
import os
import urllib.request
import glob

# 1. Chuẩn bị thư mục
JAR_DIR = "/home/iceberg/notebooks/jars_permanent"
if not os.path.exists(JAR_DIR):
    os.makedirs(JAR_DIR)

# 2. Tải file (Nếu chưa có)
libs = {
    "hadoop-aws-3.3.4.jar": "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
    "aws-java-sdk-bundle-1.12.262.jar": "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar"
}

print("🚀 Đang kiểm tra file thư viện...")
for name, url in libs.items():
    file_path = os.path.join(JAR_DIR, name)
    if not os.path.exists(file_path) or os.path.getsize(file_path) < 1024 * 1024:
        print(f"⬇️ Đang tải {name}...")
        try:
            urllib.request.urlretrieve(url, file_path)
            print(f"✅ Tải xong: {name}")
        except Exception as e:
            print(f"❌ Lỗi tải {name}: {e}")
    else:
        print(f"✅ Đã có sẵn: {name}")

# 3. TẠO CHUỖI ĐƯỜNG DẪN (KHÔNG DÙNG WILDCARD *)
jar_files = glob.glob(os.path.join(JAR_DIR, "*.jar"))
# Chuỗi dùng cho spark.jars (dấu phẩy)
jars_str = ",".join(jar_files)
# Chuỗi dùng cho classpath (dấu hai chấm trên Linux)
classpath_str = ":".join(jar_files)

print("\n🎯 ĐÃ TẠO CHUỖI CẤU HÌNH CHUẨN:")
print(f"JARs List: {jars_str}")

🚀 Đang kiểm tra file thư viện...
⬇️ Đang tải hadoop-aws-3.3.4.jar...
✅ Tải xong: hadoop-aws-3.3.4.jar
✅ Đã có sẵn: aws-java-sdk-bundle-1.12.262.jar

🎯 ĐÃ TẠO CHUỖI CẤU HÌNH CHUẨN:
JARs List: /home/iceberg/notebooks/jars_permanent/aws-java-sdk-bundle-1.12.262.jar,/home/iceberg/notebooks/jars_permanent/hadoop-aws-3.3.4.jar


In [6]:
from pyspark.sql import SparkSession
import os

# Tắt session cũ
try:
    SparkSession.builder.getOrCreate().stop()
except:
    pass

# Đảm bảo jars_str đã được tạo chính xác từ Cell 2
# jars_str = "/home/iceberg/notebooks/jars_permanent/aws-java-sdk-bundle-1.12.262.jar,/home/iceberg/notebooks/jars_permanent/hadoop-aws-3.3.4.jar"
# (Giả định jars_str đã được tạo ở Cell 2)

print(f"Sử dụng JARs: {jars_str}")

# ⚠️ Tối ưu hóa: CHỈ sử dụng spark.jars
spark = SparkSession.builder \
    .appName("SCD2_Fixed_S3A_Test") \
    .config("spark.jars", jars_str) \
    .config("spark.sql.defaultCatalog", "my_catalog") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

print("✅ Spark Session đã lên! Test đọc file ngay:")

# Test ngay lập tức (Nhớ upload file lên MinIO trước nếu bucket trống)
try:
    df = spark.read.option("header","true").csv("s3a://warehouse/raw/olist_customers_dataset.csv")
    df.show(1)
    print("🏆 THÀNH CÔNG! HỆ THỐNG ĐÃ HOẠT ĐỘNG.")
except Exception as e:
    print(f"❌ Vẫn lỗi: {e}")

Sử dụng JARs: /home/iceberg/notebooks/jars_permanent/aws-java-sdk-bundle-1.12.262.jar,/home/iceberg/notebooks/jars_permanent/hadoop-aws-3.3.4.jar


25/11/30 09:42:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/30 09:42:45 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


✅ Spark Session đã lên! Test đọc file ngay:
❌ Vẫn lỗi: An error occurred while calling o317.csv.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.Da

25/11/30 09:42:48 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://warehouse/raw/olist_customers_dataset.csv.
java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:53)
	at org.apache.spark.sql.execution.datasources.DataS

In [7]:
import socket
import urllib.request
import os

print("🔍 ĐANG KIỂM TRA KẾT NỐI INTERNET TỪ TRONG DOCKER...\n")

target_host = "repo1.maven.org"
target_url = "https://repo1.maven.org/maven2/"

# --- TEST 1: KIỂM TRA DNS (Quan trọng nhất với Java) ---
print(f"1️⃣ Test DNS Resolution cho '{target_host}':")
try:
    ip_address = socket.gethostbyname(target_host)
    print(f"   ✅ THÀNH CÔNG! IP: {ip_address}")
except Exception as e:
    print(f"   ❌ THẤT BẠI: Không thể phân giải tên miền. Lỗi DNS.")
    print(f"   👉 Nguyên nhân: Docker chưa nhận được DNS 8.8.8.8")
    print(f"   ℹ️ Chi tiết lỗi: {e}")

# --- TEST 2: KIỂM TRA TẢI DỮ LIỆU (HTTP GET) ---
print(f"\n2️⃣ Test kết nối HTTP tới '{target_url}':")
try:
    # Thử kết nối với timeout 5 giây
    response = urllib.request.urlopen(target_url, timeout=5)
    print(f"   ✅ THÀNH CÔNG! Status Code: {response.getcode()} (OK)")
    print("   👉 Kết luận: Mạng Internet hoàn toàn bình thường.")
except Exception as e:
    print(f"   ❌ THẤT BẠI: Không thể kết nối tới Server.")
    print(f"   👉 Nguyên nhân: Có thể do Firewall công ty, VPN, hoặc Docker mất mạng.")
    print(f"   ℹ️ Chi tiết lỗi: {e}")

# --- TEST 3: KIỂM TRA MÔI TRƯỜNG ---
print("\n3️⃣ Kiểm tra biến môi trường Java (IPv4):")
java_opts = os.environ.get('SPARK_DRIVER_EXTRAJAVAOPTIONS', 'Không tìm thấy')
print(f"   ℹ️ SPARK_DRIVER_EXTRAJAVAOPTIONS: {java_opts}")

🔍 ĐANG KIỂM TRA KẾT NỐI INTERNET TỪ TRONG DOCKER...

1️⃣ Test DNS Resolution cho 'repo1.maven.org':
   ✅ THÀNH CÔNG! IP: 104.18.18.12

2️⃣ Test kết nối HTTP tới 'https://repo1.maven.org/maven2/':
   ✅ THÀNH CÔNG! Status Code: 200 (OK)
   👉 Kết luận: Mạng Internet hoàn toàn bình thường.

3️⃣ Kiểm tra biến môi trường Java (IPv4):
   ℹ️ SPARK_DRIVER_EXTRAJAVAOPTIONS: Không tìm thấy
